# 05 Weather Analysis

This notebook studies race weather patterns and their relationship with lap time and race outcomes.

The objective is to decide which environmental features should be retained for Gold feature engineering.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "05_weather_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 05_weather_analysis
Start time: 2026-06-02 00:46:23.356833
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
weather = pd.read_parquet(CLEANED_DATA_PATH / "weather.parquet")
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")

sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)
session_context = sessions[["session_key", "year", "event_type", "circuit_short_name", "country_name", "date_start"]].drop_duplicates()
weather_race = weather.merge(session_context, on="session_key", how="left")
weather_race["date"] = pd.to_datetime(weather_race["date"], errors="coerce", utc=True) if "date" in weather_race.columns else pd.NaT
print(f"Weather rows: {len(weather_race):,}")
print(f"Sessions with weather: {weather_race['session_key'].nunique()}")
print(f"Circuits: {weather_race['circuit_short_name'].nunique()}")

Weather rows: 9,485
Sessions with weather: 68
Circuits: 24


## 1. Weather Variable Distributions

Weather values are continuous race-context signals. The key question is whether they have enough variation to matter for tyre and pace modeling.

In [3]:
weather_cols = [column for column in ["air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall"] if column in weather_race.columns]
weather_summary = weather_race[weather_cols].describe().round(3).reset_index().rename(columns={"index": "metric"})
weather_summary.to_csv(OUTPUT_TABLES / "weather_distribution_summary.csv", index=False)
display(weather_summary)

,metric,air_temperature,track_temperature,humidity,pressure,wind_speed,rainfall
0,count,9485.000,9485.000,9485.000,9485.000,9485.000,9485.000
1,mean,23.145,34.754,55.225,998.681,1.820,0.047
2,std,4.907,9.594,16.667,27.874,1.045,0.212
3,min,12.100,12.200,14.000,918.900,0.000,0.000
4,25%,19.000,27.300,45.000,993.000,1.100,0.000
5,50%,23.000,34.600,55.200,1009.900,1.600,0.000
6,75%,27.200,43.100,68.000,1016.400,2.400,0.000
7,max,34.100,54.600,93.000,1031.000,7.200,1.000


In [4]:
plot_cols = [column for column in ["air_temperature", "track_temperature", "humidity", "wind_speed"] if column in weather_race.columns]
plot_df = weather_race[plot_cols].melt(var_name="weather_variable", value_name="value").dropna()
fig = px.box(
    plot_df,
    x="weather_variable",
    y="value",
    color="weather_variable",
    title="Weather Variable Distributions",
)
fig.write_html(OUTPUT_CHARTS / "weather_distributions.html", include_plotlyjs="cdn")
fig.show()

## 2. Wet vs Dry Sessions

Rainfall is sparse but strategically important. This section summarizes how often wet conditions appear at session level.

In [5]:
session_weather = weather_race.groupby("session_key", as_index=False).agg(
    air_temperature=("air_temperature", "mean"),
    track_temperature=("track_temperature", "mean"),
    humidity=("humidity", "mean"),
    pressure=("pressure", "mean"),
    wind_speed=("wind_speed", "mean"),
    rainfall=("rainfall", "mean"),
    year=("year", "first"),
    event_type=("event_type", "first"),
    circuit_short_name=("circuit_short_name", "first"),
    country_name=("country_name", "first"),
    date_start=("date_start", "first"),
)
session_weather["is_wet_session"] = session_weather["rainfall"].fillna(0) > 0
wet_summary = session_weather.groupby(["year", "event_type"], as_index=False).agg(
    sessions=("session_key", "count"),
    wet_sessions=("is_wet_session", "sum"),
    wet_rate=("is_wet_session", "mean"),
)
wet_summary.to_csv(OUTPUT_TABLES / "wet_session_summary.csv", index=False)
display(wet_summary)

,year,event_type,sessions,wet_sessions,wet_rate
0,2024,GRAND_PRIX_RACE,24,5,0.208333
1,2024,SPRINT_RACE,6,1,0.166667
2,2025,GRAND_PRIX_RACE,24,6,0.250000
3,2025,SPRINT_RACE,6,2,0.333333
4,2026,GRAND_PRIX_RACE,5,1,0.200000
5,2026,SPRINT_RACE,3,0,0.000000


In [6]:
fig = px.bar(
    wet_summary,
    x="year",
    y="wet_rate",
    color="event_type",
    barmode="group",
    title="Wet Session Rate by Year and Event Type",
)
fig.write_html(OUTPUT_CHARTS / "wet_session_rate.html", include_plotlyjs="cdn")
fig.show()

## 3. Weather vs Lap Time

This joins session-level weather to session-level lap metrics to measure whether environmental conditions correlate with pace.

In [7]:
lap_session = laps.copy()
lap_session["lap_duration"] = pd.to_numeric(lap_session["lap_duration"], errors="coerce")
lap_session = lap_session[lap_session["lap_duration"].between(50, 900)]
lap_session_summary = lap_session.groupby("session_key", as_index=False).agg(
    median_lap=("lap_duration", "median"),
    fastest_lap=("lap_duration", "min"),
    lap_std=("lap_duration", "std"),
    laps=("lap_duration", "count"),
)
weather_lap = session_weather.merge(lap_session_summary, on="session_key", how="inner")
weather_lap.to_csv(OUTPUT_TABLES / "weather_lap_session_features.csv", index=False)
weather_lap_corr = weather_lap[[column for column in ["air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall", "median_lap", "fastest_lap", "lap_std"] if column in weather_lap.columns]].corr()
weather_lap_corr.to_csv(OUTPUT_TABLES / "weather_lap_correlations.csv")
display(weather_lap_corr.round(3))

,air_temperature,track_temperature,humidity,pressure,wind_speed,rainfall,median_lap,fastest_lap,lap_std
air_temperature,1.000,0.793,-0.359,-0.046,-0.162,-0.256,-0.107,-0.045,-0.225
track_temperature,0.793,1.000,-0.512,-0.006,-0.184,-0.293,-0.254,-0.192,-0.287
humidity,-0.359,-0.512,1.000,-0.074,-0.058,0.487,0.149,0.062,0.407
pressure,-0.046,-0.006,-0.074,1.000,0.177,-0.311,0.264,0.252,0.074
wind_speed,-0.162,-0.184,-0.058,0.177,1.000,-0.153,-0.021,-0.012,0.114
rainfall,-0.256,-0.293,0.487,-0.311,-0.153,1.000,0.123,0.013,0.402
median_lap,-0.107,-0.254,0.149,0.264,-0.021,0.123,1.000,0.972,0.246
fastest_lap,-0.045,-0.192,0.062,0.252,-0.012,0.013,0.972,1.000,0.136
lap_std,-0.225,-0.287,0.407,0.074,0.114,0.402,0.246,0.136,1.000


In [8]:
fig = px.scatter(
    weather_lap,
    x="track_temperature",
    y="median_lap",
    color="event_type",
    hover_name="circuit_short_name",
    title="Track Temperature vs Median Lap Time",
)
if len(weather_lap.dropna(subset=["track_temperature", "median_lap"])) > 2:
    fit_data = weather_lap.dropna(subset=["track_temperature", "median_lap"])
    fit = np.polyfit(fit_data["track_temperature"], fit_data["median_lap"], deg=1)
    x_line = np.array([fit_data["track_temperature"].min(), fit_data["track_temperature"].max()])
    fig.add_scatter(x=x_line, y=fit[0] * x_line + fit[1], mode="lines", name="Linear fit")
fig.write_html(OUTPUT_CHARTS / "track_temp_vs_lap_time.html", include_plotlyjs="cdn")
fig.show()

In [9]:
fig = px.imshow(
    weather_lap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Weather and Lap-Time Correlation Matrix",
)
fig.write_html(OUTPUT_CHARTS / "weather_lap_correlation_heatmap.html", include_plotlyjs="cdn")
fig.show()

## 4. Circuit Weather Profiles

Aggregating weather by circuit identifies venues with systematically hotter, wetter, or more humid conditions.

In [10]:
circuit_weather = session_weather.groupby("circuit_short_name", as_index=False).agg(
    avg_air_temperature=("air_temperature", "mean"),
    avg_track_temperature=("track_temperature", "mean"),
    avg_humidity=("humidity", "mean"),
    avg_pressure=("pressure", "mean"),
    avg_wind_speed=("wind_speed", "mean"),
    wet_rate=("is_wet_session", "mean"),
    sessions=("session_key", "count"),
).sort_values("avg_track_temperature", ascending=False)
circuit_weather.to_csv(OUTPUT_TABLES / "circuit_weather_profiles.csv", index=False)
display(circuit_weather.head(15))

,circuit_short_name,avg_air_temperature,avg_track_temperature,avg_humidity,avg_pressure,avg_wind_speed,wet_rate,sessions
14,Monza,29.931307,46.299736,37.444940,994.990675,1.739607,0.000000,2
20,Spielberg,29.313131,46.091781,38.117720,939.193299,1.442835,0.333333,3
12,Monte Carlo,21.906916,44.822047,58.241810,1017.761286,0.974323,0.000000,2
2,Catalunya,26.626234,44.819665,59.033682,1002.064053,1.878140,0.500000,2
0,Austin,28.268221,44.379458,40.547505,1001.850284,2.113199,0.000000,4
10,Mexico City,23.147117,43.095394,37.142441,1009.900000,1.999237,0.000000,2
4,Imola,24.539326,42.343888,43.366620,1008.217133,2.694629,0.000000,2
11,Miami,27.898076,40.814661,63.210453,1014.868131,2.006996,0.500000,6
3,Hungaroring,25.436935,38.479020,51.987026,984.402358,1.668589,0.000000,2
1,Baku,23.628922,35.228197,48.926535,1019.378079,1.529184,0.000000,2


In [11]:
fig = px.scatter(
    circuit_weather,
    x="avg_air_temperature",
    y="avg_track_temperature",
    size="wet_rate",
    color="avg_humidity",
    hover_name="circuit_short_name",
    title="Circuit Weather Profiles",
)
fig.write_html(OUTPUT_CHARTS / "circuit_weather_profiles.html", include_plotlyjs="cdn")
fig.show()

## 5. Seasonal Weather Patterns

Month-level aggregation helps reveal whether calendar placement introduces systematic weather differences.

In [12]:
session_weather["month"] = pd.to_datetime(session_weather["date_start"], errors="coerce", utc=True).dt.month
monthly_weather = session_weather.groupby("month", as_index=False).agg(
    air_temperature=("air_temperature", "mean"),
    track_temperature=("track_temperature", "mean"),
    humidity=("humidity", "mean"),
    rainfall=("rainfall", "mean"),
    sessions=("session_key", "count"),
)
monthly_weather.to_csv(OUTPUT_TABLES / "monthly_weather.csv", index=False)
display(monthly_weather)

,month,air_temperature,track_temperature,humidity,rainfall,sessions
0,3,20.342828,29.236289,48.858778,0.032584,10
1,4,21.843648,30.743848,60.575823,0.000000,6
2,5,24.432408,39.062717,56.639658,0.024752,12
3,6,26.146533,43.105411,47.682444,0.041980,7
4,7,20.302175,32.955991,64.630499,0.164164,6
5,8,20.377747,30.270439,63.062785,0.000000,3
6,9,27.581342,39.900298,49.478590,0.000000,5
7,10,26.920213,42.508464,44.407129,0.005357,7
8,11,21.519281,28.008369,59.310535,0.088905,9
9,12,24.122920,28.546868,56.379720,0.000000,3


In [13]:
fig = px.line(
    monthly_weather,
    x="month",
    y=["air_temperature", "track_temperature"],
    markers=True,
    title="Seasonal Temperature Pattern",
)
fig.write_html(OUTPUT_CHARTS / "seasonal_temperature.html", include_plotlyjs="cdn")
fig.show()

## 6. Weather vs Race Outcomes

Weather is joined to race results to inspect whether conditions correlate with finishing position, DNF-like outcomes, or points.

In [14]:
outcome = session_result.merge(session_weather, on="session_key", how="left")
outcome["finish_pos"] = pd.to_numeric(outcome["position"], errors="coerce")
outcome["is_dnf_like"] = outcome[["dnf", "dns", "dsq"]].any(axis=1) | outcome["finish_pos"].isna()
outcome_cols = [column for column in ["finish_pos", "points", "is_dnf_like"] if column in outcome.columns]
weather_vars = [column for column in ["air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall"] if column in outcome.columns]
rows = []
for weather_var in weather_vars:
    for outcome_col in outcome_cols:
        series = outcome[outcome_col].astype(float) if outcome_col == "is_dnf_like" else pd.to_numeric(outcome[outcome_col], errors="coerce")
        rows.append({"weather_var": weather_var, "outcome": outcome_col, "correlation": outcome[weather_var].corr(series)})
weather_outcome_corr = pd.DataFrame(rows)
weather_outcome_corr.to_csv(OUTPUT_TABLES / "weather_outcome_correlations.csv", index=False)
display(weather_outcome_corr.pivot(index="weather_var", columns="outcome", values="correlation").round(3))

outcome,finish_pos,is_dnf_like,points
weather_var,,,
air_temperature,0.027,-0.067,-0.020
humidity,-0.026,0.044,0.011
pressure,0.013,0.008,0.010
rainfall,-0.038,0.051,0.011
track_temperature,0.023,-0.061,0.000
wind_speed,-0.000,0.012,-0.050


In [15]:
corr_pivot = weather_outcome_corr.pivot(index="weather_var", columns="outcome", values="correlation")
fig = px.imshow(
    corr_pivot,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Weather vs Race Outcome Correlations",
)
fig.write_html(OUTPUT_CHARTS / "weather_outcome_correlation_heatmap.html", include_plotlyjs="cdn")
fig.show()

## Final Weather Analysis Report

In [16]:
temp_lap_corr = float(weather_lap["track_temperature"].corr(weather_lap["median_lap"])) if {"track_temperature", "median_lap"}.issubset(weather_lap.columns) else float("nan")
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "weather_rows": int(len(weather_race)),
    "sessions_with_weather": int(session_weather["session_key"].nunique()),
    "wet_sessions": int(session_weather["is_wet_session"].sum()),
    "wet_session_rate_pct": float(session_weather["is_wet_session"].mean() * 100),
    "avg_air_temperature": float(session_weather["air_temperature"].mean()),
    "avg_track_temperature": float(session_weather["track_temperature"].mean()),
    "track_temp_median_lap_corr": temp_lap_corr,
}
write_report("weather_analysis", report)
write_insight(
    "Silver Weather Analysis Insights",
    [
        f"Analyzed {report['sessions_with_weather']} sessions with weather data.",
        f"Wet session rate: {report['wet_session_rate_pct']:.1f}%.",
        f"Track temperature vs median lap correlation: {report['track_temp_median_lap_corr']:.3f}.",
    ],
    [],
    [
        "Keep track_temperature, air_temperature, humidity, pressure, wind_speed, and rainfall as Gold race-context candidates.",
        "Treat rainfall as sparse but strategically important rather than dropping it for low frequency.",
        "Use circuit-level weather summaries to enrich race context features.",
    ],
)
(CHECKPOINTS / "silver_weather_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '05_weather_analysis', 'timestamp': '2026-06-02T00:46:25.393511', 'weather_rows': 9485, 'sessions_with_weather': 68, 'wet_sessions': 15, 'wet_session_rate_pct': 22.058823529411764, 'avg_air_temperature': 23.3241409613225, 'avg_track_temperature': 34.862358490300274, 'track_temp_median_lap_corr': -0.254040194577628}
